# Wetgeving & Documentstroom

**Thema:** De publicatie en doorstroming van Kamerstukken en dossiers.

**Kernvragen:** Welke dossiers bevatten de meeste documenten? Wat is de doorlooptijd tussen registratie, ontvangst en definitieve publicatie van documentversies?

Dit notebook verkent o.a. de silver-tabellen `kamerstukdossier`, `document`, `document_versie`, `document_publicatie` en `document_zaak`.

**Aanpak:** dit notebook verkent de silver-tabellen `kamerstukdossier`, `document`, `document_versie`,
`document_publicatie` en `document_zaak`. Ik begin met het schema van deze tabellen, werk daarna de
twee kernvragen uit met SQL-queries en (waar mogelijk) visualisaties, daarna beantwoordt ik nog wat extra deelvragen en daarna sluit ik af met een samenvatting van de belangrijkste bevindingen en datakwaliteitsproblemen die ik onderweg ben tegengekomen. Deze
bevindingen dienen als input voor het ontwerp van de gold-laag en het dashboard.

## Setup
Ik importeer `duckdb` om queries op de lokale database uit te voeren, `pandas` om de resultaten als
DataFrame te bekijken, en `matplotlib` voor visualisaties.

In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

### Verbinding met de database
Deze functie zoekt automatisch `tweedekamer.duckdb` op door vanaf de huidige map omhoog te zoeken, zodat
de connectie werkt ongeacht vanuit welke submap het notebook gestart wordt. Ik open de database
`read_only`, omdat ik alleen data opvraag en niets wijzig. `query_df` is een korte helper die elke
SQL-query direct als pandas DataFrame teruggeeft.

In [3]:
def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

WindowsPath('C:/Users/Nynke/Documents/Bi tweedekamer opdracht/tweedekamer-monitor/eda/tweedekamer.duckdb')

## De silver-tabellen verkennen
Voordat ik ga joinen en aggregeren, is het belangrijk om te weten welke kolommen en datatypes de
relevante tabellen hebben, dit voorkomt verkeerde aannames over kolomnamen of relaties. Ik vraag het
schema daarom rechtstreeks op bij `information_schema.columns`, voor de vijf tabellen die bij dit
onderwerp horen.

In [9]:
schema = query_df("""
  SELECT table_name, column_name, data_type
  FROM information_schema.columns
  WHERE table_schema = 'silver'
    AND table_name IN ('kamerstukdossier', 'document', 'document_versie', 'document_publicatie', 'document_zaak')
  ORDER BY table_name, ordinal_position
""")
schema

,table_name,column_name,data_type
0,document,id,VARCHAR
1,document,content_type,VARCHAR
2,document,content_length,BIGINT
3,document,soort,VARCHAR
4,document,document_nummer,VARCHAR
5,document,titel,VARCHAR
6,document,onderwerp,VARCHAR
7,document,datum,DATE
8,document,volgnummer,BIGINT
9,document,vergaderjaar,VARCHAR


**Wat we hieruit leren:**
- `document` bevat zelf al `datum_registratie` en `datum_ontvangst` (deze hoeven dus niet via een andere
  tabel opgezocht te worden.)
- De koppeling dossier ↔ document loopt via `document.kamerstukdossier_id` → `kamerstukdossier.id`.
- `document_zaak` is een pure koppeltabel (alleen `document_id`, `zaak_id` en een wijzigingsdatum) (dit
  bevestigt een veel-op-veel relatie tussen documenten en zaken.
- `document_versie` heeft een eigen `datum`-kolom (de versiedatum) en een `status`-kolom, die we verderop
  gebruiken.
- Opvallend: de opdrachttekst noemt de tabel `document_publicaties` (meervoud), maar in het schema heet
  deze `document_publicatie` (enkelvoud)

## Deelvraag 1: Welke dossiers bevatten de meeste documenten?
**Aanpak:** ik tel per dossier het aantal gekoppelde documenten, door `kamerstukdossier` te joinen met
`document` op `kamerstukdossier_id`. Ik sorteer aflopend op aantal en bekijk de top 20, zodat ik snel
zie welk dossiers qua documentvolume uitspringen.

In [4]:
dossiers_meeste_documenten = query_df("""
  SELECT
    kd.id,
    kd.titel,
    kd.nummer,
    COUNT(d.id) AS aantal_documenten
  FROM silver.kamerstukdossier kd
  JOIN silver.document d
    ON d.kamerstukdossier_id = kd.id
  GROUP BY kd.id, kd.titel, kd.nummer
  ORDER BY aantal_documenten DESC
  LIMIT 20
""")
dossiers_meeste_documenten

,id,titel,nummer,aantal_documenten
0,60f20417-bf77-4e74-bcef-101ee574ca47,Nieuwe Commissievoorstellen en initiatieven va...,22112,3783
1,6328c8bd-f8d2-4d24-87fe-15ad845492bc,Raad Algemene Zaken en Raad Buitenlandse Zaken,21501,2635
2,7eb61ff4-c6c3-476f-8123-fd5394fd0e1a,Vreemdelingenbeleid,19637,2420
3,54da79c7-a208-48c7-8014-6852f9a36089,Infectieziektenbestrijding,25295,2272
4,0eccfde6-9d74-4bf8-9fdb-9af73f6d8063,Europese Raad,21501,2041
5,a8701ea8-fb89-4579-ac33-cf2de468f6f8,Raad voor Economische en Financiële Zaken,21501,1597
6,44d5d174-d387-49d5-a25b-28b1f9c878f4,Kabinetsaanpak Klimaatbeleid,32813,1576
7,cc39c6da-829a-4b57-bd32-b6264d68f7f1,Landbouw- en Visserijraad,21501,1561
8,c162a19a-5a21-4927-9f15-8534750862de,Belastingdienst,31066,1525
9,a32ce8ad-2b6f-447d-9d5a-d8167c25fbca,Integrale visie op de woningmarkt,32847,1501


**Wat opvalt:** het dossier "Nieuwe Commissievoorstellen en initiatieven van de lidstaten van de Europese
Unie" (nummer 22112) ligt met 3783 documenten met afstand voor op de rest. Verschillende van de top-dossiers
gaan over EU-raden (Raad Algemene Zaken, Europese Raad, Economische/Financiële Raad, Landbouw- en
Visserijraad), dit soort dossiers loopt blijkbaar al jaren door en verzamelt daardoor veel losse
verslagen/brieven. Dit verschilt duidelijk van eenmalige of korter lopende dossiers.

**Technische kanttekening:** bij het plotten van deze resultaten crashte de Jupyter-kernel herhaaldelijk,
ook bij een minimale, losstaande matplotlib-test. Om de deadline te halen is besloten de resultaten als tabel te tonen in plaats van als grafiek,
de top-20 hierboven laat het verschil tussen dossiers ook prima zien zonder plot.

## Deelvraag 2: Wat is de doorlooptijd tussen registratie, ontvangst en definitieve publicatie?
**Aanpak:** ik koppel `document` (registratie-/ontvangstdatum) via `document_versie` aan
`document_publicatie` (publicatiedatum), en bereken het aantal dagen tussen elk paar momenten.
Voordat ik dat doe, check ik eerst welke waarden de kolom `status` in `document_versie` kan hebben,
omdat de kernvraag spreekt over "definitieve" publicatie.

In [13]:
statussen = query_df("""
  SELECT status, COUNT(*) AS aantal
  FROM silver.document_versie
  GROUP BY status
  ORDER BY aantal DESC
""")
statussen

,status,aantal
0,Vrijgegeven,611885


Er blijkt maar één status-waarde voor te komen: **"Vrijgegeven"**, voor alle 611.885 rijen. Er is dus geen
onderscheid zoals "concept" of "definitief" in deze kolom, filteren op status is daarom niet nodig.
Blijkbaar worden niet-vrijgegeven versies dus niet (of nog
niet) in deze silver-tabel opgenomen.

In [11]:
doorlooptijd = query_df("""
  SELECT
    d.id AS document_id,
    d.datum_registratie,
    d.datum_ontvangst,
    dv.id AS document_versie_id,
    dv.datum AS datum_versie,
    dp.publicatie_datum,
    DATE_DIFF('day', d.datum_registratie, d.datum_ontvangst) AS dagen_registratie_naar_ontvangst,
    DATE_DIFF('day', d.datum_ontvangst, dp.publicatie_datum) AS dagen_ontvangst_naar_publicatie,
    DATE_DIFF('day', d.datum_registratie, dp.publicatie_datum) AS dagen_registratie_naar_publicatie
  FROM silver.document d
  JOIN silver.document_versie dv
    ON dv.document_id = d.id
  JOIN silver.document_publicatie dp
    ON dp.document_versie_id = dv.id
  WHERE d.datum_registratie IS NOT NULL
    AND d.datum_ontvangst IS NOT NULL
    AND dp.publicatie_datum IS NOT NULL
""")
doorlooptijd.describe()

,datum_registratie,datum_ontvangst,datum_versie,publicatie_datum,dagen_registratie_naar_ontvangst,dagen_ontvangst_naar_publicatie,dagen_registratie_naar_publicatie
count,191635,191635,191635,191635,191635.000000,191635.000000,191635.000000
mean,2022-07-09 05:09:54.699297,2022-07-14 15:04:34.177472,2022-08-10 23:01:16.100920,2025-05-31 15:07:20.092884,5.412957,1052.001920,1057.414877
min,2016-05-17 00:00:00,2016-05-18 00:00:00,2018-10-17 00:00:00,2025-01-06 00:00:00,-625.000000,-179.000000,0.000000
25%,2020-09-08 00:00:00,2020-09-15 00:00:00,2020-10-01 00:00:00,2025-04-28 00:00:00,1.000000,348.000000,357.000000
50%,2022-06-10 00:00:00,2022-06-14 00:00:00,2022-08-31 00:00:00,2025-04-28 00:00:00,2.000000,1049.000000,1053.000000
75%,2024-05-06 00:00:00,2024-05-15 00:00:00,2024-05-17 00:00:00,2025-04-28 00:00:00,5.000000,1686.000000,1693.000000
max,2026-09-07 00:00:00,2026-09-15 00:00:00,2026-09-25 00:00:00,2026-09-25 00:00:00,512.000000,3264.000000,3265.000000
std,NaN,NaN,NaN,NaN,13.917913,754.572316,754.462680


**Wat opvalt:**
- Er zijn negatieve waarden bij beide doorlooptijd-kolommen. Zo is de minimale `dagen_registratie_naar_ontvangst`
  **-625**, dat betekent dat bij sommige documenten de ontvangstdatum vóór de registratiedatum ligt, wat niet
  logisch is.
- `dagen_ontvangst_naar_publicatie` heeft een minimum van **-179**, publicatie vóór ontvangst is nog
  vreemder, en ook dit wijst op een datakwaliteitsprobleem.
- Bij `publicatie_datum` hebben het 25e, 50e én 75e percentiel **exact dezelfde datum** (2025-04-28). Dat is
  zeer onwaarschijnlijk bij echte, individuele publicatiemomenten, dit wordt hieronder verder onderzocht.
- Hierdoor is de gemiddelde doorlooptijd tot publicatie extreem hoog (gemiddeld **1052 dagen**, bijna 3 jaar), een onrealistische waarde voor een "doorlooptijd".

In [12]:
# aantal documenten gepubliceert op publicatiedatum: 
publicatiedatums_verdeling = query_df("""
  SELECT
    dp.publicatie_datum,
    COUNT(*) AS aantal
  FROM silver.document_publicatie dp
  GROUP BY dp.publicatie_datum
  ORDER BY aantal DESC
  LIMIT 15
""")
publicatiedatums_verdeling

,publicatie_datum,aantal
0,2025-04-28,489421
1,2025-04-25,33868
2,2025-03-20,8312
3,2025-05-14,1076
4,2025-04-04,884
5,2026-03-10,860
6,2025-05-21,857
7,2025-03-27,820
8,2025-10-03,712
9,2025-09-22,696


**Bevestiging van het vermoeden:** maar liefst **489.421** van de ~611.885 documentpublicaties (ruim 80%)
hebben exact dezelfde `publicatie_datum`: **28 april 2025**. Dit is vrijwel zeker geen toeval, maar het gevolg
van een bulk-verwerking: een groot deel van de historische documenten is op dat ene moment in het systeem
gepubliceerd, waarbij `publicatie_datum` de datum van die verwerking lijkt
te zijn geworden, niet de oorspronkelijke publicatiedatum.

**Consequentie voor deze deelvraag:** `publicatie_datum` is in de huidige vorm **niet betrouwbaar** als maat
voor de werkelijke doorlooptijd tot publicatie, voor het grootste deel van de data. Dit is een belangrijke
bevinding om mee te geven aan het ontwerp van de gold-laag, deze kolom zou niet zonder correctie gebruikt
moeten worden voor een doorlooptijd-KPI op het dashboard. Als alternatief voor "wanneer een document
daadwerkelijk klaar was" zou `document_versie.datum` (de versiedatum) bruikbaarder kunnen zijn, omdat die
kolom een veel realistischere spreiding laat zien (gemiddelde rond augustus 2022, geen opvallende
samenklontering op één datum).

## Deelvraag 3: Hoe lang blijven dossiers actief?
De tabel `kamerstukdossier` heeft geen eigen start-/einddatum, dus ik benader de looptijd van een dossier via de datums van de bijbehorende
documenten: het verschil tussen het vroegste en laatste document-datum binnen een dossier. Oorspronkelijk
was het plan om deze looptijd te vergelijken tussen afgesloten en nog-openstaande dossiers (via de kolom
`afgesloten`), maar dit bleek niet mogelijk, zie de bevinding hieronder: 

In [14]:
dossier_looptijd = query_df("""
  SELECT
    kd.id,
    kd.titel,
    kd.nummer,
    kd.afgesloten,
    MIN(d.datum) AS eerste_document_datum,
    MAX(d.datum) AS laatste_document_datum,
    COUNT(d.id) AS aantal_documenten,
    DATE_DIFF('day', MIN(d.datum), MAX(d.datum)) AS looptijd_dagen
  FROM silver.kamerstukdossier kd
  JOIN silver.document d
    ON d.kamerstukdossier_id = kd.id
  WHERE d.datum IS NOT NULL
  GROUP BY kd.id, kd.titel, kd.nummer, kd.afgesloten
""")
dossier_looptijd.describe()

,nummer,eerste_document_datum,laatste_document_datum,aantal_documenten,looptijd_dagen
count,7521.000000,7521,7521,7521.000000,7521.000000
mean,33921.486371,2016-01-13 09:38:24.746709,2018-01-08 03:35:00.837654,28.306475,725.747640
min,17050.000000,1988-06-13 00:00:00,1991-09-19 00:00:00,1.000000,0.000000
25%,32548.000000,2010-11-30 00:00:00,2012-09-20 00:00:00,3.000000,22.000000
50%,34210.000000,2015-06-01 00:00:00,2018-02-08 00:00:00,7.000000,120.000000
75%,35688.000000,2020-12-22 00:00:00,2023-04-13 00:00:00,15.000000,444.000000
max,37035.000000,2026-09-17 00:00:00,2026-09-25 00:00:00,3783.000000,10787.000000
std,2358.072895,NaN,NaN,114.702739,1540.310685


In [19]:
looptijd_per_status = query_df("""
  SELECT
    afgesloten,
    COUNT(*) AS aantal_dossiers,
    AVG(looptijd_dagen) AS gemiddelde_looptijd_dagen,
    MEDIAN(looptijd_dagen) AS mediaan_looptijd_dagen,
    MAX(looptijd_dagen) AS langste_looptijd_dagen
  FROM (
    SELECT
      kd.id,
      kd.afgesloten,
      DATE_DIFF('day', MIN(d.datum), MAX(d.datum)) AS looptijd_dagen
    FROM silver.kamerstukdossier kd
    JOIN silver.document d ON d.kamerstukdossier_id = kd.id
    WHERE d.datum IS NOT NULL
    GROUP BY kd.id, kd.afgesloten
  ) t
  GROUP BY afgesloten
""")
looptijd_per_status

,afgesloten,aantal_dossiers,gemiddelde_looptijd_dagen,mediaan_looptijd_dagen,langste_looptijd_dagen
0,False,7521,725.74764,120.0,10787


In [17]:
langstlopende_dossiers = query_df("""
  SELECT
    kd.id,
    kd.titel,
    kd.nummer,
    MIN(d.datum) AS eerste_document_datum,
    MAX(d.datum) AS laatste_document_datum,
    DATE_DIFF('day', MIN(d.datum), MAX(d.datum)) / 365.25 AS looptijd_jaren
  FROM silver.kamerstukdossier kd
  JOIN silver.document d ON d.kamerstukdossier_id = kd.id
  WHERE d.datum IS NOT NULL
    AND kd.afgesloten = true
  GROUP BY kd.id, kd.titel, kd.nummer
  ORDER BY looptijd_jaren DESC
  LIMIT 15
""")
langstlopende_dossiers

,id,titel,nummer,eerste_document_datum,laatste_document_datum,looptijd_jaren


**Wat opvalt:**
- De mediaan looptijd van een dossier is **120 dagen**, maar het gemiddelde ligt veel hoger op **725,7 dagen**, dit grote verschil wijst op een scheve verdeling: een klein aantal extreem langlopende dossiers trekt hetgemiddelde flink omhoog.
- De langstlopende dossiers lopen tot **10.787 dagen** (bijna 30 jaar), met eerste documenten die al uit 1988
  dateren.
- 25% van de dossiers heeft een looptijd van **22 dagen of minder**, er is dus een duidelijke tweedeling
  tussen veel kortlopende/eenmalige dossiers en een kleinere groep dossiers die al tientallen jaren actief is.

**Datakwaliteitsbevinding:** de kolom `afgesloten` staat voor **alle 7792 dossiers** in de tabel op `False`, 
er is dus geen enkel dossier dat formeel als afgesloten gemarkeerd is. Hierdoor kon de geplande
vergelijking tussen afgesloten en open dossiers niet uitgevoerd worden. Dit kan erop wijzen dat deze kolom
niet correct wordt bijgewerkt in het ETL-proces, of dat de brondata dit onderscheid simpelweg niet maakt.
Voor het dashboard/gold-laag betekent dit dat `afgesloten` in de huidige vorm **niet bruikbaar** is als
filter of indicator, tenzij de oorzaak hiervan wordt opgehelderd.

## Deelvraag 4: Verschilt de documentstroom per aanleverende organisatie?
Ik wil onderzoeken welke organisaties de meeste documenten aanleveren, en of de doorlooptijd tussen registratie
en ontvangst verschilt per organisatie. Dit bleek een minder rijke vraag dan verwacht: de kolom `organisatie`
bevat vrijwel geen variatie (zie bevindingen hieronder), waardoor een zinvolle vergelijking tussen organisaties
niet mogelijk is.

In [20]:
documenten_per_organisatie = query_df("""
  SELECT
    organisatie,
    COUNT(*) AS aantal_documenten
  FROM silver.document
  WHERE organisatie IS NOT NULL
  GROUP BY organisatie
  ORDER BY aantal_documenten DESC
  LIMIT 20
""")
documenten_per_organisatie

,organisatie,aantal_documenten
0,Tweede Kamer,609539
1,Eerste Kamer en Tweede Kamer,2345
2,0,1


In [21]:
doorlooptijd_per_organisatie = query_df("""
  SELECT
    organisatie,
    COUNT(*) AS aantal_documenten,
    AVG(DATE_DIFF('day', datum_registratie, datum_ontvangst)) AS gemiddelde_dagen_registratie_naar_ontvangst,
    MEDIAN(DATE_DIFF('day', datum_registratie, datum_ontvangst)) AS mediane_dagen_registratie_naar_ontvangst
  FROM silver.document
  WHERE organisatie IS NOT NULL
    AND datum_registratie IS NOT NULL
    AND datum_ontvangst IS NOT NULL
  GROUP BY organisatie
  HAVING COUNT(*) >= 100
  ORDER BY gemiddelde_dagen_registratie_naar_ontvangst DESC
""")
doorlooptijd_per_organisatie

,organisatie,aantal_documenten,gemiddelde_dagen_registratie_naar_ontvangst,mediane_dagen_registratie_naar_ontvangst
0,Tweede Kamer,110589,3.241100,1.0
1,Eerste Kamer en Tweede Kamer,1291,1.112316,1.0


In [22]:
document_organisatie_nul = query_df("""
  SELECT *
  FROM silver.document
  WHERE organisatie = '0'
""")
document_organisatie_nul

,id,content_type,content_length,soort,document_nummer,titel,onderwerp,datum,volgnummer,vergaderjaar,...,organisatie,kamerstukdossier_id,huidige_document_versie_id,kenmerk_afzender,aanhangselnummer,alias,citeertitel,document_url,gewijzigd_op,api_gewijzigd_op
0,4e1cf6bc-f710-4f7d-81b1-7671ebb6cfd7,application/pdf,27494,Schriftelijke vragen,2024D41483,None,Het bericht 'Einde van het melktappunt’,2024-10-31,-1,2024-2025,...,0,None,ac7b1f44-bd19-4bdb-93ea-31cd9c2a4682,None,None,None,None,https://gegevensmagazijn.tweedekamer.nl/SyncFe...,2024-12-02 08:47:01.420000+01:00,2024-12-02 07:48:05.034755+01:00


**Wat opvalt:**
- De kolom `organisatie` bevat slechts 3 unieke waarden, en is sterk gedomineerd door de Tweede Kamer
  (609.539 van de ~611.885 documenten, ruim 99,5%). Er is dus feitelijk geen diversiteit aan aanleverende
  organisaties binnen deze dataset om een doorlooptijd-vergelijking op te baseren.
- Tussen de twee bruikbare waarden is er bovendien nauwelijks verschil in doorlooptijd: "Tweede Kamer" heeft
  een gemiddelde van 3,24 dagen (mediaan 1,0), "Eerste Kamer en Tweede Kamer" een gemiddelde van 1,11 dagen
  (ook mediaan 1,0). De oorspronkelijke verwachting (verschil tussen organisaties) klopt
  dus niet, en dat is zelf de bevinding.
- Opvallende datakwaliteitsvondst: één document (`2024D41483`) heeft meerdere verdachte waarden tegelijk:
  `organisatie = '0'`, `volgnummer = -1`, en een ontbrekende `titel` (wel een gevuld `onderwerp`-veld:
  "Het bericht 'Einde van het melktappunt'", een Schriftelijke Vraag). De combinatie van een letterlijke "0",
  een negatief volgnummer en een lege titel wijst op een document dat ergens in de bron of het ETL-proces
  incompleet is verwerkt, eerder dan op toeval.
- Conclusie: `organisatie` is in deze tabel niet geschikt als vergelijkingsdimensie, het veld bevat vrijwel
  uitsluitend "Tweede Kamer" en voegt in de huidige vorm weinig analytische waarde toe. 

## Deelvraag 5: Verschilt de documentstroom tussen politieke fracties?
Een voor de hand liggende vervolgvraag op de organisatie-analyse is of er verschil zit tussen de
politieke fracties die documenten indienen. De tabellen die ik tot nu toe gebruikte bevatten geen directe
fractie-informatie, dus eerst zoek ik uit welke silver-tabellen daarvoor in aanmerking komen.

In [30]:
alle_silver_tabellen = query_df("""
  SELECT table_name
  FROM information_schema.tables
  WHERE table_schema = 'silver'
  ORDER BY table_name
""")
alle_silver_tabellen

,table_name
0,activiteit
1,activiteit_actor
2,activiteit_vervangen_vanuit
3,activiteit_voortgezet_vanuit
4,agendapunt
5,besluit
6,besluit_zaak
7,commissie
8,commissie_contactinformatie
9,commissie_zetel


Uit het volledige overzicht van silver-tabellen springen twee namen eruit: `document_actor` (een
koppeltabel tussen documenten en de personen/fracties die erbij betrokken zijn) en `fractie` (de
partijgegevens zelf). Ik bekijk hun schema om de juiste join-kolommen te vinden.

In [24]:
schema_actor_fractie = query_df("""
  SELECT table_name, column_name, data_type
  FROM information_schema.columns
  WHERE table_schema = 'silver'
    AND table_name IN ('document_actor', 'fractie')
  ORDER BY table_name, ordinal_position
""")
schema_actor_fractie

,table_name,column_name,data_type
0,document_actor,id,VARCHAR
1,document_actor,actor_naam,VARCHAR
2,document_actor,functie,VARCHAR
3,document_actor,relatie,VARCHAR
4,document_actor,sid_actor,VARCHAR
5,document_actor,persoon_id,VARCHAR
6,document_actor,document_id,VARCHAR
7,document_actor,actor_fractie,VARCHAR
8,document_actor,fractie_id,VARCHAR
9,document_actor,gewijzigd_op,TIMESTAMP WITH TIME ZONE


`document_actor.fractie_id` blijkt een directe foreign key naar `fractie.id` te zijn, de kortste,
betrouwbaarste route. Belangrijk is de kolom `relatie`: die geeft de rol van de actor bij het document aan
(bijv. "Indiener" of "Ondertekenaar"). Omdat één document meerdere actoren/fracties kan hebben, check ik
eerst welke waarden deze kolom kan hebben, om dubbeltellingen te voorkomen.

In [25]:
relatie_waarden = query_df("""
  SELECT relatie, COUNT(*) AS aantal
  FROM silver.document_actor
  GROUP BY relatie
  ORDER BY aantal DESC
""")
relatie_waarden

,relatie,aantal
0,Eerste ondertekenaar,419379
1,Mede ondertekenaar,155784
2,Mede namens,17440
3,Afzender,195
4,Geaddresseerde,160
5,Namens,138
6,Mede afzender,8


Er is een duidelijke hoofdrol: **"Eerste ondertekenaar"** (419.379 van de ~611.885 rijen), naast
nevenrollen zoals "Mede ondertekenaar" en "Mede namens". Voor een eerlijke vergelijking tussen fracties
filteren we daarom op alleen `relatie = 'Eerste ondertekenaar', zo telt elk document maar bij één fractie
mee (de daadwerkelijke hoofdindiener), in plaats van bij elke fractie die ook maar ergens bij het document
betrokken was.

In [27]:
documenten_per_fractie_hoofdindiener = query_df("""
  SELECT
    f.naam_nl,
    f.afkorting,
    COUNT(DISTINCT da.document_id) AS aantal_documenten
  FROM silver.document_actor da
  JOIN silver.fractie f
    ON da.fractie_id = f.id
  WHERE da.relatie = 'Eerste ondertekenaar'
  GROUP BY f.naam_nl, f.afkorting
  ORDER BY aantal_documenten DESC
  LIMIT 20
""")
documenten_per_fractie_hoofdindiener

,naam_nl,afkorting,aantal_documenten
0,Volkspartij voor Vrijheid en Democratie,VVD,26627
1,Socialistische Partij,SP,25328
2,Partij van de Arbeid,PvdA,21327
3,Christen-Democratisch Appèl,CDA,18778
4,Democraten 66,D66,18561
5,Partij voor de Vrijheid,PVV,13715
6,GroenLinks,GL,11467
7,ChristenUnie,ChristenUnie,8054
8,Partij voor de Dieren,PvdD,7035
9,GroenLinks-PvdA,GroenLinks-PvdA,4581


Na filtering op hoofdindiener blijven er 8 fracties over met 100+ documenten. Dit is een realistischer
beeld dan een eerdere, ongefilterde telling (niet in dit notebook opgenomen) die door dubbeltelling een
vertekend beeld gaf. Vervolgens onderzoek ik of de doorlooptijd tussen registratie en ontvangst (de
enige doorlooptijd-maat die ik bij Deelvraag 2 betrouwbaar genoeg vond) verschilt per fractie.

In [28]:
doorlooptijd_per_fractie = query_df("""
  SELECT
    f.naam_nl,
    f.afkorting,
    COUNT(DISTINCT d.id) AS aantal_documenten,
    AVG(DATE_DIFF('day', d.datum_registratie, d.datum_ontvangst)) AS gemiddelde_dagen_registratie_naar_ontvangst,
    MEDIAN(DATE_DIFF('day', d.datum_registratie, d.datum_ontvangst)) AS mediane_dagen_registratie_naar_ontvangst
  FROM silver.document d
  JOIN silver.document_actor da
    ON da.document_id = d.id
  JOIN silver.fractie f
    ON da.fractie_id = f.id
  WHERE da.relatie = 'Eerste ondertekenaar'
    AND d.datum_registratie IS NOT NULL
    AND d.datum_ontvangst IS NOT NULL
  GROUP BY f.naam_nl, f.afkorting
  HAVING COUNT(DISTINCT d.id) >= 100
  ORDER BY gemiddelde_dagen_registratie_naar_ontvangst DESC
""")
doorlooptijd_per_fractie

,naam_nl,afkorting,aantal_documenten,gemiddelde_dagen_registratie_naar_ontvangst,mediane_dagen_registratie_naar_ontvangst
0,Partij voor de Vrijheid,PVV,280,4.275000,1.5
1,GroenLinks-PvdA,GroenLinks-PvdA,101,4.198020,1.0
2,Volkspartij voor Vrijheid en Democratie,VVD,1638,2.772894,1.0
3,Democraten 66,D66,655,2.717557,0.0
4,Partij van de Arbeid,PvdA,792,1.573232,0.0
5,Christen-Democratisch Appèl,CDA,589,0.534805,1.0
6,Socialistische Partij,SP,287,-5.066202,0.0
7,GroenLinks,GL,246,-37.016260,0.0


**Wat opvalt:** de gemiddelde doorlooptijd verschilt behoorlijk per fractie, van 4,3 dagen (PVV) tot
**-37,0 dagen** (GroenLinks) en -5,1 dagen (SP). Zo'n sterk negatief gemiddelde bij GroenLinks valt op,
zeker vergeleken met de andere fracties die allemaal rond de 0 tot 4 dagen zitten. Ik wil dieper ingaan op de
meest extreme GroenLinks-documenten om te zien of dit een breed patroon is of door een paar uitschieters
komt.

In [29]:
gl_negatieve_doorlooptijd = query_df("""
  SELECT
    d.id,
    d.titel,
    d.datum_registratie,
    d.datum_ontvangst,
    DATE_DIFF('day', d.datum_registratie, d.datum_ontvangst) AS dagen_registratie_naar_ontvangst
  FROM silver.document d
  JOIN silver.document_actor da
    ON da.document_id = d.id
  JOIN silver.fractie f
    ON da.fractie_id = f.id
  WHERE da.relatie = 'Eerste ondertekenaar'
    AND f.afkorting = 'GL'
    AND d.datum_registratie IS NOT NULL
    AND d.datum_ontvangst IS NOT NULL
  ORDER BY dagen_registratie_naar_ontvangst ASC
  LIMIT 10
""")
gl_negatieve_doorlooptijd

,id,titel,datum_registratie,datum_ontvangst,dagen_registratie_naar_ontvangst
0,0897e85b-d03b-4f91-ac18-1f1ae7171198,Voorstel van het lid Halsema tot wijziging van...,2014-04-07,2005-03-09,-3316
1,7de5ebf0-ef98-4248-bda0-e1cc0fa099bf,Voorstel van het lid Halsema tot wijziging van...,2014-04-07,2005-03-09,-3316
2,aa7d30cb-5023-4a90-bc82-8a4620a1f197,Voorstel van het lid Halsema tot wijziging van...,2014-04-07,2007-01-25,-2629
3,f2497210-a5ee-45d2-8a0f-23d857b0efa5,Parlementair onderzoek Lessen uit recente arbe...,2011-11-08,2011-10-13,-26
4,7fd8cc30-1d36-4c24-9dd6-8e31866ce767,Uitvoering Wet Werk en Bijstand,2010-08-24,2010-08-17,-7
5,218ffbc0-062b-4b1e-9bf4-df3c20f23d1b,Vrij verkeer werknemers uit de nieuwe EU lidst...,2010-09-13,2010-09-09,-4
6,3cd7481c-655b-4486-9a50-bea6b9a90646,Duurzame ontwikkeling en beleid,2008-09-08,2008-09-04,-4
7,2843a08e-80fe-4c3d-9a74-532fff7f8431,Voorstel van wet van de leden Voortman en Van ...,2014-10-14,2014-10-14,0
8,2359e428-8226-4c56-a661-5e81cbc15b7e,Initiatiefnota van het lid Westerveld: ons lan...,2022-12-20,2022-12-20,0
9,7fa05553-e535-4b6e-986a-406af5a6dfca,Voorstel van wet van het lid Maatoug tot wijzi...,2022-01-26,2022-01-26,0


**Wat opvalt:**
- Het blijkt **niet** een breed patroon, maar het gevolg van een paar extreme uitschieters: drie documenten
  rond hetzelfde initiatiefwetsvoorstel van het lid Halsema hebben een `datum_ontvangst` die 7-9 jaar vóór
  de `datum_registratie` ligt. De overige 7 van de 10 meest extreme gevallen liggen tussen -26 en 0 dagen.
- Extra datakwaliteitsvondst: twee van deze documenten hebben een **identieke titel, registratiedatum én
  ontvangstdatum, maar een verschillend `id`**, een aanwijzing voor mogelijke duplicaten in de `document`-
  tabel.
- Conclusie: fractiegemiddeldes zijn gevoelig voor een klein aantal extreme uitschieters bij fracties met
  relatief weinig documenten (zoals GroenLinks met 246). De mediaan (die bij GroenLinks wél 0,0 is) is hier
  een betere maat dan het gemiddelde.

## Samenvatting en bevindingen

**Deelvraag 1: meeste documenten per dossier:** langlopende, overkoepelende EU-gerelateerde dossiers
(zoals "Nieuwe Commissievoorstellen...", Raad Algemene Zaken, Europese Raad) bevatten verreweg de meeste
documenten, met als koploper dossier 22112 (3783 documenten).

**Deelvraag 2: doorlooptijd registratie/ontvangst/publicatie:** de doorlooptijd van registratie naar
ontvangst is meestal kort (mediaan 2 dagen), maar met negatieve uitschieters die op datafouten wijzen. De
doorlooptijd tot `publicatie_datum` is onbruikbaar als betrouwbare maat: deze kolom wordt gedomineerd door
een vermoedelijke bulk-verwerking op 28 april 2025 (ruim 80% van de rijen deelt deze exacte datum).

**Deelvraag 3: hoe lang blijven dossiers actief:** de mediane looptijd van een dossier is 120 dagen, maar
het gemiddelde ligt veel hoger (725,7 dagen) door een kleine groep extreem langlopende dossiers (tot bijna
30 jaar). De geplande vergelijking tussen afgesloten en open dossiers bleek niet uitvoerbaar: de kolom
`afgesloten` staat voor alle 7792 dossiers op `False`.

**Deelvraag 4: verschil per aanleverende organisatie:** de kolom `organisatie` bevat vrijwel geen
variatie (99,5% "Tweede Kamer") en is daardoor niet geschikt als vergelijkingsdimensie. Wel leverde dit een
concrete datakwaliteitsvondst op: één document met de verdachte combinatie `organisatie = '0'`,
`volgnummer = -1` en een ontbrekende titel.

**Deelvraag 5: verschil per politieke fractie:** na correctie voor dubbeltelling (filteren op
`relatie = 'Eerste ondertekenaar'`) verschilt de gemiddelde doorlooptijd per fractie, met GroenLinks als
sterke negatieve uitschieter (-37,0 dagen). Nader onderzoek laat zien dat dit door een paar extreme
documenten komt, niet door een structureel verschil tussen fracties, de mediaan ligt voor GroenLinks
gewoon op 0,0.

**Datakwaliteitsproblemen, samengevat:**
1. Negatieve doorlooptijden tussen registratie/ontvangst (tot -625 dagen) en ontvangst/publicatie (tot
   -179 dagen).
2. Massale samenklontering van `publicatie_datum` op één datum (28 april 2025), wat wijst op een
   bulk-herverwerking in plaats van individuele publicatiemomenten.
3. De kolom `afgesloten` in `kamerstukdossier` bevat geen enkele `True`-waarde over de volledige tabel.
4. De kolom `organisatie` in `document` bevat vrijwel geen variatie en bevat minstens één duidelijk
   foutieve waarde (`'0'`).
5. Mogelijke duplicaten in de `document`-tabel: meerdere records met identieke titel, registratie- en
   ontvangstdatum maar verschillend `id`.
6. Naamsverschil tussen opdracht (`document_publicaties`) en werkelijk schema (`document_publicatie`).

**Suggesties voor de gold-laag:**
- Gebruik `document_versie.datum` in plaats van `document_publicatie.publicatie_datum` als maat voor
  "wanneer een document gereed/vrijgegeven was", tenzij de oorzaak van de bulk-datum wordt opgehelderd.
- Laat `kamerstukdossier.afgesloten` buiten een eventueel gold-dimensiemodel totdat duidelijk is waarom
  deze kolom nooit `True` is.
- Overweeg een aparte check/ontdubbeling op (titel, datum_registratie, datum_ontvangst) in `document` om
  mogelijke duplicaten te signaleren vóórdat deze doorstromen naar gold.
- Fractie-gebaseerde metrics (zoals gemiddelde doorlooptijd) zijn gevoelig voor uitschieters bij fracties
  met weinig documenten, gebruik bij voorkeur de mediaan, of toon beide waarden naast elkaar op het
  dashboard.